Changing Colorspaces 
====================

Goal
----

-   In this tutorial, you will learn how to convert images from one color-space to another, like
    BGR $\leftrightarrow$ Gray, BGR $\leftrightarrow$ HSV, etc.
-   In addition to that, we will create an application to extract a colored object in a video
-   You will learn the following functions: **cv.cvtColor()**, **cv.inRange()**, etc.

Changing Color-space
--------------------

There are more than 150 color-space conversion methods available in OpenCV. But we will look into
only two, which are most widely used ones: BGR $\leftrightarrow$ Gray and BGR $\leftrightarrow$ HSV.

For color conversion, we use the function cv.cvtColor(input_image, flag) where flag determines the
type of conversion.

For BGR $\rightarrow$ Gray conversion, we use the flag cv.COLOR_BGR2GRAY. Similarly for BGR
$\rightarrow$ HSV, we use the flag cv.COLOR_BGR2HSV. To get other flags, just run following
commands in your Python terminal:

In [ ]:
import cv2 as cv

flags = [i for i in dir(cv) if i.startswith("COLOR_")]
print(flags)

> **Note:** For HSV, hue range is [0,179], saturation range is [0,255], and value range is [0,255].
Different software use different scales. So if you are comparing OpenCV values with them, you need
to normalize these ranges.

Object Tracking
---------------

Now that we know how to convert a BGR image to HSV, we can use this to extract a colored object. In HSV, it
is easier to represent a color than in BGR color-space. In our application, we will try to extract
a blue colored object. So here is the method:

-   Take each frame of the video
-   Convert from BGR to HSV color-space
-   We threshold the HSV image for a range of blue color
-   Now extract the blue object alone, we can do whatever we want on that image.

Below is the code which is commented in detail:

In [ ]:
import asyncio

import cv2 as cv
import ipywidgets as widgets
import numpy as np
from ipycanvas import Canvas, hold_canvas

# Open the video
cap = cv.VideoCapture("bottle-detection.mp4")
total_frames = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv.CAP_PROP_FPS) or 25

ret, frame = cap.read()
if not ret:
    raise RuntimeError("Could not read video")

frame = cv.resize(frame, (400, 300))
frame_h, frame_w = frame.shape[:2]

# ---------- interactive widgets ----------
play_btn = widgets.ToggleButton(value=False, description="Play", icon="play")
frame_sl = widgets.IntSlider(
    value=0, min=0, max=max(total_frames - 1, 0), description="Frame"
)

h_low = widgets.IntSlider(value=0, min=0, max=179, description="H low")
h_high = widgets.IntSlider(value=179, min=0, max=179, description="H high")
s_low = widgets.IntSlider(value=0, min=0, max=255, description="S low")
s_high = widgets.IntSlider(value=255, min=0, max=255, description="S high")
v_low = widgets.IntSlider(value=0, min=0, max=255, description="V low")
v_high = widgets.IntSlider(value=255, min=0, max=255, description="V high")

# ---------- canvases ----------
c_frame = Canvas(width=frame_w, height=frame_h)
c_mask = Canvas(width=frame_w, height=frame_h)
c_res = Canvas(width=frame_w, height=frame_h)

# ---------- state ----------
_current_frame = frame.copy()
_play_task = None


def process_and_draw(f):
    hsv = cv.cvtColor(f, cv.COLOR_BGR2HSV)
    lower = np.array([h_low.value, s_low.value, v_low.value])
    upper = np.array([h_high.value, s_high.value, v_high.value])
    mask = cv.inRange(hsv, lower, upper)
    res = cv.bitwise_and(f, f, mask=mask)
    with hold_canvas():
        for c, img in [(c_frame, f), (c_mask, mask), (c_res, res)]:
            c.put_image_data(
                cv.cvtColor(img, cv.COLOR_BGR2RGB) if img.ndim == 3 else img, 0, 0
            )


def update(*_):
    """Refresh canvases from the current frame (called by slider callbacks)."""
    process_and_draw(_current_frame)


async def playback_loop():
    while True:
        ret, f = cap.read()
        if not ret:
            cap.set(cv.CAP_PROP_POS_FRAMES, 0)
            continue
        f = cv.resize(f, (frame_w, frame_h))
        _current_frame[...] = f
        frame_sl.value = min(int(cap.get(cv.CAP_PROP_POS_FRAMES)), total_frames - 1)
        process_and_draw(f)
        await asyncio.sleep(1.0 / fps)


for w in (h_low, h_high, s_low, s_high, v_low, v_high):
    w.observe(update, "value")


def on_play(change):
    global _play_task
    play_btn.description = "Pause" if change["new"] else "Play"
    play_btn.icon = "pause" if change["new"] else "play"
    if change["new"]:
        _play_task = asyncio.create_task(playback_loop())
    elif _play_task is not None:
        _play_task.cancel()
        _play_task = None


def on_frame_slider(change):
    if not play_btn.value:  # only seek when paused
        cap.set(cv.CAP_PROP_POS_FRAMES, change["new"])
        ret, f = cap.read()
        if ret:
            f = cv.resize(f, (frame_w, frame_h))
            _current_frame[...] = f
            process_and_draw(f)


play_btn.observe(on_play, "value")
frame_sl.observe(on_frame_slider, "value")

update()

display(
    widgets.VBox(
        [
            widgets.HBox([play_btn, frame_sl]),
            widgets.HBox([h_low, h_high]),
            widgets.HBox([s_low, s_high]),
            widgets.HBox([v_low, v_high]),
            widgets.HBox(
                [c_frame, c_mask, c_res],
                layout=widgets.Layout(justify_content="space-around"),
            ),
        ]
    )
)

> **Note:** There is some noise in the image. We will see how to remove it in later chapters.

> **Note:** This is the simplest method in object tracking. Once you learn functions of contours, you can
do plenty of things like find the centroid of an object and use it to track the object, draw diagrams
just by moving your hand in front of a camera, and other fun stuff.

How to find HSV values to track?
--------------------------------

This is a common question found in [stackoverflow.com](http://www.stackoverflow.com). It is very simple and
you can use the same function, cv.cvtColor(). Instead of passing an image, you just pass the BGR
values you want. For example, to find the HSV value of Green, try the following commands in a Python
terminal:

In [ ]:
green = np.uint8([[[0, 255, 0]]])
hsv_green = cv.cvtColor(green, cv.COLOR_BGR2HSV)
hsv_green

Now you take [H-10, 100,100] and [H+10, 255, 255] as the lower bound and upper bound respectively. Apart
from this method, you can use any image editing tools like GIMP or any online converters to find
these values, but don't forget to adjust the HSV ranges.

Exercises
---------

1. Try to find a way to extract more than one colored object, for example, extract red, blue, and green
objects simultaneously.